# Imports

In [ ]:
import os
os.environ.setdefault("PYTORCH_ENABLE_MPS_FALLBACK", "1")

import dataclasses
import json
import math
import resource
import sys
import time

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
from tqdm.auto import tqdm

from uavloc.data.annotated import AnnotatedImage
from uavloc.datasets import dataset_ref
from uavloc.features.segmentation import build_segmenter
from uavloc.segmentation.eval import VAL_SETS, load_val_sets
from uavloc.segmentation.losses import CURRENT_LOSS, StructureLossConfig, structure_loss_terms
from uavloc.segmentation.monitor import crop_metrics, tiled_crops
from uavloc.segmentation.preview import log_segmenter_previews
import uavloc.segmentation.village_seg as vs
from uavloc.track import experiment_dir, log_history, log_hparams, resolve_device, set_seed, tensorboard_writer

# Params

In [ ]:
EXP = "033-class-loss-in-system"
ARM = "control"                       # control | building | structure | noise
LOSS = "current"                      # current (CE + Dice) | building | structure (building + track terms)
SEED = 0
STEPS = 3000
EVAL_EVERY = 250                      # log point: BN recalibration, train/val monitor, previews, running loss
DEVICE = "auto"                       # cuda if present, else mps, else cpu
OUT_DIR = ""                          # empty means artifacts/segmenter-ARM, under the experiment folder

GAP_WEIGHT = 3.0
GAP_RADIUS_PX = 10
INSTANCE_WEIGHT = 1.0
INSTANCE_REF_PX = 1046
INSTANCE_MAX_RATIO = 10.0
TVERSKY_WEIGHT = 1.0
TVERSKY_ALPHA = 0.3
TVERSKY_BETA = 0.7
SKELETON_WEIGHT = 1.0
SKELETON_DILATION_PX = 1
TRACK_TVERSKY_WEIGHT = 1.0
TRACK_TVERSKY_ALPHA = 0.3
TRACK_TVERSKY_BETA = 0.7

DRONE_P = 0.5
DRONE_GSD_M = 0.10
SAT_CROP_PX, SAT_BATCH = 512, 4
DRONE_CROP_PX, DRONE_BATCH = 256, 8
LR = 5e-4
WEIGHT_DECAY = 1e-4
WARMUP_FRAC = 0.05
RARE_CROP_P = 0.3
BN_BATCHES = 60
BN_SEED = 123
MONITOR_SEED = 11
MONITOR_TRAIN_CROPS = 90              # satellite; the drone side gets as many crops as its val tiles
EVAL_BATCH = 8
PREVIEW_CROPS = 4

In [ ]:
ARMS = ("control", "building", "structure", "noise")
LOSSES = ("current", "building", "structure")
DOMAINS = ("satellite", "drone")
DRONE_PREFIX = "drone"                # drone metrics are logged as train/drone_<name>, val/drone_<name>
SPLITS = ("train", "val")
KIB, GIB = 1024, 2 ** 30

# Data

In [ ]:
if ARM not in ARMS or LOSS not in LOSSES:
    raise ValueError(f"ARM must be one of {ARMS} and LOSS one of {LOSSES}, got {ARM!r} / {LOSS!r}")
set_seed(SEED)
device = resolve_device(DEVICE)
out_dir = experiment_dir(EXP) / (OUT_DIR or f"artifacts/segmenter-{ARM}")
out_dir.mkdir(parents=True, exist_ok=True)
device, out_dir.name

In [ ]:
split = vs.SplitConfig()
data = vs.load_village_data(split)
vs.check_round3_splits(data)
{"sat_train": len(data.sat_train), "drone_train": len(data.drone_train), "drone_val": len(data.drone_val)}

In [ ]:
samplers = {"satellite": vs.SatelliteSampler(data.sat_train, SAT_CROP_PX, RARE_CROP_P),
            "drone": vs.DroneSampler(data.drone_train, DRONE_GSD_M, DRONE_CROP_PX, RARE_CROP_P)}
batch_sizes = {"satellite": SAT_BATCH, "drone": DRONE_BATCH}

In [ ]:
val_sets = load_val_sets(split.dataset)
{name: len(images) for name, images in val_sets.items()}

In [ ]:
def drone_val_images() -> list[AnnotatedImage]:
    """Val frames resampled to the training GSD, the scale the model is trained to see."""
    return [AnnotatedImage(d.name, "", *vs.drone_at_gsd(d, DRONE_GSD_M)) for d in data.drone_val]

In [ ]:
val_crops = {"satellite": tiled_crops([a for name in VAL_SETS for a in val_sets[name]], SAT_CROP_PX),
             "drone": tiled_crops(drone_val_images(), DRONE_CROP_PX)}
{domain: tuple(x.shape) for domain, (x, _) in val_crops.items()}

In [ ]:
def train_crops(domain: str, n: int) -> tuple[torch.Tensor, torch.Tensor]:
    """`n` fixed training-distribution crops of one domain (seeded draw from its sampler)."""
    return vs.batch_of(samplers[domain], np.random.default_rng(MONITOR_SEED), n)

In [ ]:
monitor = {"satellite": {"train": train_crops("satellite", MONITOR_TRAIN_CROPS), "val": val_crops["satellite"]},
           "drone": {"train": train_crops("drone", len(val_crops["drone"][0])), "val": val_crops["drone"]}}
{d: {s: tuple(x.shape) for s, (x, _) in splits.items()} for d, splits in monitor.items()}

In [ ]:
def evenly_spaced(crops: tuple[torch.Tensor, torch.Tensor], n: int) -> tuple[torch.Tensor, torch.Tensor]:
    """`n` evenly spaced crops of a set, so the preview slider compares the same pixels all run."""
    idx = np.linspace(0, len(crops[0]) - 1, n).round().astype(int)
    return crops[0][idx], crops[1][idx]

In [ ]:
previews = {domain: {split: evenly_spaced(crops, PREVIEW_CROPS) for split, crops in splits.items()}
            for domain, splits in monitor.items()}

# Model

In [ ]:
building_terms = dict(gap_weight=GAP_WEIGHT, gap_radius_px=GAP_RADIUS_PX, instance_weight=INSTANCE_WEIGHT,
                      instance_ref_px=INSTANCE_REF_PX, instance_max_ratio=INSTANCE_MAX_RATIO,
                      tversky_weight=TVERSKY_WEIGHT, tversky_alpha=TVERSKY_ALPHA, tversky_beta=TVERSKY_BETA)
track_terms = dict(skeleton_weight=SKELETON_WEIGHT, skeleton_dilation_px=SKELETON_DILATION_PX,
                   track_tversky_weight=TRACK_TVERSKY_WEIGHT, track_tversky_alpha=TRACK_TVERSKY_ALPHA,
                   track_tversky_beta=TRACK_TVERSKY_BETA)
loss_config = {"current": CURRENT_LOSS, "building": StructureLossConfig(**building_terms),
               "structure": StructureLossConfig(**building_terms, **track_terms)}[LOSS]
loss_config

In [ ]:
model = build_segmenter("lraspp-mobilenetv3", pretrained=True).to(device)
n_params = sum(p.numel() for p in model.parameters())
opt = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
sched = torch.optim.lr_scheduler.OneCycleLR(opt, max_lr=LR, total_steps=STEPS, pct_start=WARMUP_FRAC)
n_params

In [ ]:
def loss_terms(logits: torch.Tensor, y: torch.Tensor) -> dict[str, torch.Tensor]:
    """Every term of the structure loss; `loss` is the one this run optimises (all arms, control included)."""
    return structure_loss_terms(logits, y, loss_config)

In [ ]:
def training_step(domain: str, rng: np.random.Generator) -> dict[str, float]:
    """One optimiser step on a fresh batch of `domain`; returns every loss term as a float."""
    x, y = vs.batch_of(samplers[domain], rng, batch_sizes[domain])
    terms = loss_terms(model(x.to(device)), y.to(device))
    opt.zero_grad()
    terms["loss"].backward()
    opt.step()
    sched.step()
    return {k: v.item() for k, v in terms.items()}

In [ ]:
def bn_pair(net: torch.nn.Module) -> dict[str, torch.nn.Module]:
    """Same weights, BatchNorm statistics re-estimated per domain from its own training crops."""
    return {d: vs.domain_bn(net, samplers[d], batch_sizes[d], BN_BATCHES, BN_SEED, device) for d in DOMAINS}

In [ ]:
def monitor_entry(models: dict[str, torch.nn.Module], step: int) -> dict[str, float]:
    """Every loss term and metric on the fixed train and val crops, each domain through its own BN model."""
    entry = {"step": step}
    for domain, splits in monitor.items():
        for split, crops in splits.items():
            prefix = split if domain == "satellite" else f"{split}_{DRONE_PREFIX}"
            entry.update(crop_metrics(models[domain], crops, loss_terms, EVAL_BATCH, device, prefix))
    return entry

In [ ]:
def log_previews(models: dict[str, torch.nn.Module], step: int, final: bool = False) -> None:
    """Fixed-crop grids: `<split>/segmentation` for satellite, `<split>/drone_segmentation` for drone."""
    suffix = "_final" if final else ""
    for domain, tag in (("satellite", "segmentation"), ("drone", f"{DRONE_PREFIX}_segmentation")):
        log_segmenter_previews(writer, models[domain], previews[domain], step, device, tag + suffix)

In [ ]:
def running_means(running: list[dict[str, float]], by_domain: dict[str, list[float]]) -> dict[str, float]:
    """Optimiser-batch means since the last log point (train mode, augmented), never under a plain `train_<name>`."""
    means = {f"train_running_{k}": float(np.mean([r[k] for r in running])) for k in running[0]}
    return {**means, **{f"train_running_loss_{d}": float(np.mean(v)) for d, v in by_domain.items() if v}}

# Run

In [ ]:
writer = tensorboard_writer(f"{EXP}/segmenter-{ARM}")
rng = np.random.default_rng(SEED)
model.train()
models = bn_pair(model)
history = [monitor_entry(models, 0)]
log_history(writer, history[-1])
log_previews(models, 0)
running, by_domain, t0 = [], {d: [] for d in DOMAINS}, time.time()
progress = tqdm(range(1, STEPS + 1), desc=f"{ARM} ({LOSS}, seed {SEED})")
for step in progress:
    lr = sched.get_last_lr()[0]
    model.train()
    domain = "drone" if rng.random() < DRONE_P else "satellite"
    terms = training_step(domain, rng)
    if not math.isfinite(terms["loss"]):
        raise FloatingPointError(f"non-finite loss {terms['loss']} at step {step} ({domain} batch)")
    running.append(terms)
    by_domain[domain].append(terms["loss"])
    progress.set_postfix(loss=f"{terms['loss']:.3f}", val_miou=f"{history[-1]['val_miou']:.3f}",
                         val_drone_miou=f"{history[-1]['val_drone_miou']:.3f}")
    if step % EVAL_EVERY == 0 or step == STEPS:
        t_eval = time.time()
        models = bn_pair(model)
        entry = monitor_entry(models, step)
        history.append({**entry, **running_means(running, by_domain), "train_lr": lr,
                        "eval_minutes": (time.time() - t_eval) / 60, "minutes": (time.time() - t0) / 60})
        log_history(writer, history[-1])
        log_previews(models, step)
        running, by_domain = [], {d: [] for d in DOMAINS}
train_minutes = (time.time() - t0) / 60
f"{train_minutes:.1f} min"

In [ ]:
log_previews(models, STEPS, final=True)
torch.save(model.state_dict(), out_dir / "model.pt")
for domain, net in models.items():
    torch.save(net.state_dict(), out_dir / f"model-bn-{domain}.pt")

In [ ]:
satellite_metrics = vs.val_metrics(models["satellite"], val_sets, device)
drone_metrics = vs.drone_val_metrics(models["drone"], data.drone_val, DRONE_GSD_M, device)
{k: round(satellite_metrics[k], 4) for k in ("miou_mean", "building_missed_share_mean", "separable_merge_rate_mean",
                                             "track_skeleton_recall_mean", "track_skeleton_precision_mean")}, drone_metrics["miou_drone-val"]

In [ ]:
def memory_gb() -> dict[str, float]:
    """Peak resident memory of this process and, on MPS, what the driver holds for it."""
    rss_bytes = resource.getrusage(resource.RUSAGE_SELF).ru_maxrss * (1 if sys.platform == "darwin" else KIB)
    return {"peak_rss_gb": rss_bytes / GIB,
            "device_memory_gb": torch.mps.driver_allocated_memory() / GIB if device == "mps" else 0.0}

# Plots

In [ ]:
frame = pd.DataFrame(history)
CURVES = [("loss", "loss (optimised)"), ("ce", "CE"), ("dice", "Dice"), ("tversky_building", "Tversky, building"),
          ("gap_ce", "CE on gap band"), ("building_ce", "CE on buildings"), ("miou", "mIoU"),
          ("building_missed_share", "missed buildings"), ("skeleton_recall_track", "skeleton recall loss, track"),
          ("tversky_track", "Tversky, track"), ("track_skeleton_recall", "track centreline recall"),
          ("track_skeleton_precision", "track centreline precision")]

In [ ]:
def plot_curves(domain_prefix: str, title: str) -> plt.Figure:
    """Train vs val for every curve in `CURVES`; `domain_prefix` is "" (satellite) or "drone_"."""
    fig, axes = plt.subplots(3, 4, figsize=(18, 10))
    for ax, (key, name) in zip(axes.ravel(), CURVES):
        for split in SPLITS:
            ax.plot(frame["step"], frame[f"{split}_{domain_prefix}{key}"], "-o", ms=3, label=split)
        ax.set_title(name); ax.set_xlabel("step"); ax.grid(alpha=.25); ax.legend(fontsize=8)
    fig.suptitle(f"{title}: {ARM}, loss {LOSS}, seed {SEED}")
    fig.tight_layout()
    return fig

In [ ]:
fig_satellite = plot_curves("", "satellite")
fig_satellite.savefig(out_dir / "curves-satellite.png", dpi=90)

In [ ]:
fig_drone = plot_curves(f"{DRONE_PREFIX}_", "drone")
fig_drone.savefig(out_dir / "curves-drone.png", dpi=90)

# Log

In [ ]:
config = {"experiment": EXP, "arm": ARM, "loss": LOSS, "loss_config": vars(loss_config), "seed": SEED, "steps": STEPS,
          "eval_every": EVAL_EVERY, "device": device, "architecture": "lraspp-mobilenetv3", "init": "imagenet",
          "lr": LR, "weight_decay": WEIGHT_DECAY, "warmup_frac": WARMUP_FRAC, "drone_p": DRONE_P,
          "drone_gsd_m": DRONE_GSD_M, "sat_crop_px": SAT_CROP_PX, "sat_batch": SAT_BATCH,
          "drone_crop_px": DRONE_CROP_PX, "drone_batch": DRONE_BATCH, "rare_crop_p": RARE_CROP_P,
          "bn_batches": BN_BATCHES, "bn_seed": BN_SEED, "monitor_seed": MONITOR_SEED,
          "monitor_train_crops": MONITOR_TRAIN_CROPS, "split": dataclasses.asdict(split),
          "dataset": dataset_ref(split.dataset), "drone_dataset": dataset_ref(split.drone_dataset)}

In [ ]:
metrics = {**satellite_metrics, **drone_metrics, "train_minutes": train_minutes, "n_params": n_params,
           **memory_gb(),
           **{f"{k}_final": v for k, v in history[-1].items() if k.startswith(("train_", "val_"))}}

In [ ]:
frame.to_csv(out_dir / "history.csv", index=False)
(out_dir / "metrics.json").write_text(json.dumps(metrics, indent=1, default=str))
(out_dir / "config.json").write_text(json.dumps(config, indent=1, default=str))
log_hparams(writer, config, metrics)
out_dir